# Phase 7 — Advanced Transformations

**Duration:** Weeks 11–12 | **Goal:** Master complex data types and transformations used in real-world messy data.

---

### What We'll Cover

| Section | Key Skills |
| --- | --- |
| 7.1 Complex Types | Arrays, Maps, Structs — nested data structures in DataFrames |
| 7.2 Explode & Flatten | `explode`, `explode_outer`, `posexplode` — one row → many rows |
| 7.3 Higher-Order Functions | `transform`, `filter`, `aggregate` — lambda-like ops on arrays |
| 7.4 Pivot & Unpivot | Rows → columns and columns → rows |
| 7.5 JSON Handling | `from_json`, `to_json`, `get_json_object`, `schema_of_json` |
| 7.6 String & Regex | `regexp_extract`, `regexp_replace`, `split`, complex parsing |
| 7.7 Date & Timestamp | Timezones, intervals, date arithmetic, formatting |

### Dataset

* **Primary:** `samples.tpch.lineitem` (30M rows) + custom nested data structures
* **Supporting:** Hand-crafted DataFrames to demonstrate nested/complex scenarios

---

**Pre-requisite:** Phases 2–6 completed. This is the FINAL PySpark skills phase before the project course.

In [0]:
# ============================================================
# STEP 1: Setup — create DataFrames with complex/nested structures
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

SCHEMA = "arnalladatabricks.pyspark_learning"

# --- Dataset 1: E-commerce orders with nested items (Array of Structs) ---
orders_data = [
    (1, "Alice", "2024-03-15", [{"product": "Laptop", "qty": 1, "price": 999.99}, {"product": "Mouse", "qty": 2, "price": 29.99}], {"city": "Mumbai", "state": "MH", "zip": "400001"}),
    (2, "Bob", "2024-03-16", [{"product": "Phone", "qty": 1, "price": 699.99}], {"city": "Delhi", "state": "DL", "zip": "110001"}),
    (3, "Charlie", "2024-03-16", [{"product": "Tablet", "qty": 1, "price": 449.99}, {"product": "Case", "qty": 1, "price": 39.99}, {"product": "Charger", "qty": 2, "price": 19.99}], {"city": "Bangalore", "state": "KA", "zip": "560001"}),
    (4, "Diana", "2024-03-17", [{"product": "Monitor", "qty": 2, "price": 349.99}], {"city": "Mumbai", "state": "MH", "zip": "400002"}),
    (5, "Eve", "2024-03-17", [], {"city": "Chennai", "state": "TN", "zip": "600001"}),  # empty items!
]

orders_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer", StringType()),
    StructField("order_date", StringType()),
    StructField("items", ArrayType(StructType([
        StructField("product", StringType()),
        StructField("qty", IntegerType()),
        StructField("price", DoubleType())
    ]))),
    StructField("address", StructType([
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("zip", StringType())
    ]))
])

df_orders = spark.createDataFrame(orders_data, orders_schema)
print("ORDERS (nested: array of structs + struct):")
df_orders.printSchema()
df_orders.show(truncate=False)

# --- Dataset 2: User tags (Map type) ---
tags_data = [
    (1, "Alice", {"role": "admin", "team": "engineering", "level": "senior"}),
    (2, "Bob", {"role": "analyst", "team": "data", "level": "mid"}),
    (3, "Charlie", {"role": "engineer", "team": "engineering"}),  # missing "level"
    (4, "Diana", {}),  # empty map
]

df_tags = spark.createDataFrame(tags_data, ["user_id", "name", "tags"])
print("\nUSER TAGS (Map type):")
df_tags.printSchema()
df_tags.show(truncate=False)

# --- Dataset 3: JSON strings (common in real data) ---
json_data = [
    (1, '{"event": "login", "device": {"type": "mobile", "os": "iOS"}, "metadata": ["app_v2", "dark_mode"]}'),
    (2, '{"event": "purchase", "device": {"type": "desktop", "os": "Windows"}, "metadata": ["web_v3"]}'),
    (3, '{"event": "logout", "device": {"type": "mobile", "os": "Android"}, "metadata": []}'),
]

df_json = spark.createDataFrame(json_data, ["id", "raw_json"])
print("\nRAW JSON STRINGS (need parsing):")
df_json.show(truncate=False)

print("\n✅ All datasets ready!")

## 7.1 — Complex Types: Arrays, Maps, Structs

### The 3 complex column types:

| Type | What it holds | Example | Access syntax |
| --- | --- | --- | --- |
| **Struct** | Named fields (like a dict/object) | `{city: "Mumbai", state: "MH"}` | `col("address.city")` |
| **Array** | Ordered list of same-type elements | `["Laptop", "Mouse", "Charger"]` | `col("items")[0]`, `explode()` |
| **Map** | Key-value pairs (like a dictionary) | `{role: "admin", team: "eng"}` | `col("tags")["role"]` |

### When you see these in real data:

* **Struct:** Address fields, nested metadata, API responses
* **Array:** Order line items, user permissions, event tags
* **Map:** Dynamic key-value configs, custom attributes, labels

### Accessing nested data:

```python
# Struct: dot notation
df.select("address.city", "address.state")

# Array: index or explode
df.select(col("items")[0].alias("first_item"))  # first element
df.select(explode("items"))                      # one row per element

# Map: bracket notation
df.select(col("tags")["role"].alias("role"))     # get value by key
df.select(map_keys("tags"), map_values("tags"))  # all keys / all values
```

In [0]:
# ============================================================
# STEP 2: Accessing nested data — Struct, Array, Map
# ============================================================

print("1. STRUCT ACCESS (dot notation):")
df_orders.select(
    "order_id", "customer",
    col("address.city").alias("city"),
    col("address.state").alias("state")
).show()

print("2. ARRAY ACCESS (use try_element_at — returns NULL for empty arrays):")
df_orders.select(
    "order_id", "customer",
    expr("try_element_at(items, 1)").alias("first_item"),  # 1-indexed! NULL if empty/out-of-bounds
    size("items").alias("num_items")                        # array length (0 for empty)
).show(truncate=False)

print("3. MAP ACCESS (by key):")
df_tags.select(
    "name",
    col("tags")["role"].alias("role"),
    col("tags")["team"].alias("team"),
    col("tags")["level"].alias("level"),       # NULL if key missing
    size("tags").alias("num_tags"),
    map_keys(col("tags")).alias("all_keys")
).show(truncate=False)

print("→ Struct = dot notation | Array = index or explode | Map = bracket notation")

### Deep Dive: When to Use Which Type

**Struct vs Map — they look similar but behave differently:**

| | Struct | Map |
| --- | --- | --- |
| Schema | Fixed fields (known at compile time) | Dynamic keys (can vary per row) |
| Access | `col("field.subfield")` | `col("field")["key"]` |
| Performance | Faster (columnar storage, stats) | Slower (no column stats, no pushdown) |
| Use when | You KNOW the field names upfront | Fields vary per row (user-defined attributes) |

**Example:**
* Address (city, state, zip) → **Struct** (always the same 3 fields)
* User preferences (arbitrary key=value pairs) → **Map** (different users have different keys)

**Arrays in Delta/Parquet:**
* Stored as repeated groups
* Can’t filter/pushdown INTO arrays efficiently
* Best practice: if you always query one element, `explode` into rows or extract into a column

## 7.2 — Explode & Flatten: One Row → Many Rows

### The `explode` family:

| Function | What it does | Empty array behavior |
| --- | --- | --- |
| `explode(col)` | One row per array element | Drops the row entirely |
| `explode_outer(col)` | Same, but keeps rows with empty/null arrays | Keeps row with NULL |
| `posexplode(col)` | Like explode but adds position index | Drops row |
| `posexplode_outer(col)` | Like explode_outer + position | Keeps row with NULL |

### Visual:

```
BEFORE explode:                    AFTER explode(items):
| order_id | items          |      | order_id | item     |
| 1        | [Laptop,Mouse] |  →   | 1        | Laptop   |
|          |                |      | 1        | Mouse    |
| 2        | [Phone]        |      | 2        | Phone    |
| 5        | []             |      | (dropped!)           |
```

### `flatten` — for nested arrays (array of arrays):
```python
# [[1,2], [3,4]] → [1, 2, 3, 4]
df.select(flatten(col("nested_array")))
```

In [0]:
# ============================================================
# STEP 3: Explode — flatten arrays into rows
# ============================================================

print("1. EXPLODE (one row per item, drops empty arrays):")
df_exploded = df_orders.select(
    "order_id", "customer",
    explode("items").alias("item")
)
df_exploded.show(truncate=False)
print(f"   Notice: order_id=5 (Eve, empty items) is GONE!\n")

print("2. EXPLODE_OUTER (keeps rows with empty/null arrays):")
df_exploded_outer = df_orders.select(
    "order_id", "customer",
    explode_outer("items").alias("item")
)
df_exploded_outer.show(truncate=False)
print(f"   Now Eve appears with item=NULL\n")

print("3. POSEXPLODE (adds position index):")
df_orders.filter(col("order_id") == 3).select(
    "order_id", "customer",
    posexplode("items").alias("position", "item")
).show(truncate=False)

print("4. ACCESS STRUCT FIELDS after explode:")
df_exploded.select(
    "order_id", "customer",
    col("item.product"),
    col("item.qty"),
    col("item.price"),
    (col("item.qty") * col("item.price")).alias("line_total")
).show()

print("→ explode + struct access = the standard way to flatten nested JSON/API data")

## 7.3 — Higher-Order Functions: Transform Arrays Without Exploding

### The problem with explode:

Explode creates many rows → you do your operation → then you `collect_list` back. 3 steps!

**Higher-order functions** do it in ONE step — apply a function to each array element without changing row count.

### The 3 main functions:

| Function | What it does | SQL equivalent |
| --- | --- | --- |
| `transform(array, func)` | Apply function to EACH element | `TRANSFORM(arr, x -> x * 2)` |
| `filter(array, func)` | Keep only elements matching condition | `FILTER(arr, x -> x > 10)` |
| `aggregate(array, start, merge, finish)` | Reduce array to single value | `AGGREGATE(arr, 0, (acc, x) -> acc + x)` |

### SQL syntax (often cleaner):
```sql
SELECT
  order_id,
  TRANSFORM(items, x -> x.price * x.qty) AS line_totals,
  FILTER(items, x -> x.price > 100) AS expensive_items,
  AGGREGATE(items, 0D, (acc, x) -> acc + x.price * x.qty) AS order_total
FROM orders
```

In [0]:
# ============================================================
# STEP 4: Higher-order functions — transform arrays in place
# ============================================================

# Register as temp view for SQL higher-order functions
df_orders.createOrReplaceTempView("orders")

print("1. TRANSFORM — calculate line total for each item:")
df_transformed = spark.sql("""
    SELECT order_id, customer,
        TRANSFORM(items, x -> STRUCT(x.product, x.qty, x.price, x.qty * x.price AS line_total)) 
        AS items_with_totals
    FROM orders
    WHERE size(items) > 0
""")
df_transformed.show(truncate=False)

print("2. FILTER — keep only items priced > $100:")
df_filtered = spark.sql("""
    SELECT order_id, customer,
        FILTER(items, x -> x.price > 100) AS expensive_items
    FROM orders
""")
df_filtered.show(truncate=False)

print("3. AGGREGATE — sum all line totals into order total:")
df_totals = spark.sql("""
    SELECT order_id, customer,
        AGGREGATE(items, DOUBLE(0), (acc, x) -> acc + x.qty * x.price) AS order_total
    FROM orders
""")
df_totals.show()

print("4. PYTHON API equivalent (using transform):")
from pyspark.sql.functions import transform

df_orders.filter(size("items") > 0).select(
    "order_id",
    transform(col("items"), lambda x: x["price"] * x["qty"]).alias("line_totals")
).show(truncate=False)

print("→ Higher-order functions = no explode needed, keeps one row per order!")

### Deep Dive: Explode vs Higher-Order Functions

**When to use which:**

| Scenario | Use |
| --- | --- |
| Need to JOIN array elements with another table | `explode` (need separate rows to join) |
| Need to filter/transform and keep array structure | Higher-order functions |
| Need aggregation across arrays from DIFFERENT rows | `explode` + `groupBy` |
| Need per-row array operations (no cross-row) | Higher-order functions |

**Performance:** Higher-order functions are faster because:
* No row multiplication (explode 10 items = 10x more rows)
* No shuffle needed to re-aggregate
* Processed within a single row

**Limitation:** Higher-order functions can’t reference other columns inside the lambda in all cases. For complex logic, explode + re-collect is safer.

**`collect_list` / `collect_set` — the inverse of explode:**
```python
# After exploding and transforming, put rows back into an array:
df_exploded.groupBy("order_id").agg(
    collect_list("item").alias("items_back")
)
```

## 7.4 — Pivot & Unpivot: Reshape Your Data

### Pivot = Rows → Columns

```
BEFORE (long format):              AFTER pivot:
| region | quarter | revenue |     | region | Q1   | Q2   | Q3   |
| East   | Q1      | 100     |     | East   | 100  | 150  | 200  |
| East   | Q2      | 150     |  →  | West   | 80   | 120  | 180  |
| East   | Q3      | 200     |
| West   | Q1      | 80      |
```

```python
df.groupBy("region").pivot("quarter").sum("revenue")
```

### Unpivot = Columns → Rows (the reverse)

```python
# Databricks SQL:
SELECT * FROM table
UNPIVOT (value FOR quarter IN (Q1, Q2, Q3))

# PySpark:
df.unpivot("region", ["Q1", "Q2", "Q3"], "quarter", "revenue")
# OR use stack():
df.select("region", expr("stack(3, 'Q1', Q1, 'Q2', Q2, 'Q3', Q3) AS (quarter, revenue)"))
```

### When to use:
* **Pivot:** When you need a cross-tab / wide format for dashboards or reports
* **Unpivot:** When you receive wide Excel data and need long format for analysis

In [0]:
# ============================================================
# STEP 5: Pivot and Unpivot
# ============================================================

# --- Create sample data for pivot ---
df = spark.read.table("samples.tpch.lineitem")

print("1. PIVOT — revenue by shipmode per returnflag:")
df_pivot = df.groupBy("l_returnflag") \
    .pivot("l_shipmode") \
    .agg(round(sum("l_extendedprice") / 1_000_000, 1).alias("rev_millions"))
df_pivot.show()
print("   Each shipmode became a column!\n")

print("2. PIVOT with specific values (faster — Spark doesn't scan for distinct):")
df_pivot2 = df.groupBy("l_returnflag") \
    .pivot("l_shipmode", ["AIR", "SHIP", "TRUCK"]) \
    .agg(round(sum("l_extendedprice") / 1_000_000, 1))
df_pivot2.show()

print("3. UNPIVOT — turn pivot result back to long format:")
df_unpivot = df_pivot2.unpivot(
    ids=["l_returnflag"],
    values=["AIR", "SHIP", "TRUCK"],
    variableColumnName="shipmode",
    valueColumnName="revenue_millions"
)
df_unpivot.orderBy("l_returnflag", "shipmode").show()
print("→ Pivot = wide (good for dashboards) | Unpivot = long (good for analysis)")

## 7.5 — JSON Handling: Parse, Extract, Build

### Common scenario:

You receive data with a STRING column containing raw JSON (from APIs, Kafka, logs).

### Key functions:

| Function | What it does |
| --- | --- |
| `from_json(col, schema)` | Parse JSON string → Struct column |
| `to_json(col)` | Struct/Map/Array → JSON string |
| `get_json_object(col, path)` | Extract one value by JSONPath |
| `json_tuple(col, keys...)` | Extract multiple values at once |
| `schema_of_json(example)` | Infer schema from a JSON string |

### Pattern:
```python
# 1. Infer schema from a sample
schema = schema_of_json('{"event": "x", "device": {"type": "y"}}')

# 2. Parse the JSON column
df.withColumn("parsed", from_json(col("raw_json"), schema))

# 3. Access fields
df.select("parsed.event", "parsed.device.type")
```

In [0]:
# ============================================================
# STEP 6: JSON handling — parse, extract, build
# ============================================================

print("RAW DATA (JSON as string):")
df_json.show(truncate=False)

# --- Method 1: get_json_object (quick extraction, no schema needed) ---
print("1. get_json_object — extract specific fields by path:")
df_json.select(
    "id",
    get_json_object("raw_json", "$.event").alias("event"),
    get_json_object("raw_json", "$.device.type").alias("device_type"),
    get_json_object("raw_json", "$.device.os").alias("os"),
    get_json_object("raw_json", "$.metadata[0]").alias("first_meta")
).show()

# --- Method 2: from_json (parse entire structure) ---
print("2. from_json — parse into a proper Struct:")

# Define schema (or use schema_of_json to infer)
json_schema = StructType([
    StructField("event", StringType()),
    StructField("device", StructType([
        StructField("type", StringType()),
        StructField("os", StringType())
    ])),
    StructField("metadata", ArrayType(StringType()))
])

df_parsed = df_json.withColumn("parsed", from_json(col("raw_json"), json_schema))
df_parsed.select(
    "id",
    col("parsed.event").alias("event"),
    col("parsed.device.type").alias("device_type"),
    col("parsed.device.os").alias("os"),
    col("parsed.metadata").alias("metadata")
).show(truncate=False)

# --- Method 3: schema_of_json (auto-infer schema) ---
print("3. schema_of_json — auto-infer from a sample:")
sample_json = '{"event": "login", "device": {"type": "mobile", "os": "iOS"}, "metadata": ["app_v2"]}'
inferred_schema = schema_of_json(lit(sample_json))
print(f"   Inferred: {spark.range(1).select(inferred_schema).collect()[0][0]}")

# --- Method 4: to_json (struct back to JSON string) ---
print("\n4. to_json — convert struct back to JSON string:")
df_orders.select(
    "order_id",
    to_json(col("address")).alias("address_json")
).show(truncate=False)

print("→ get_json_object = quick & dirty | from_json = proper parsing with schema")

## 7.6 — String & Regex: Parsing Messy Text

### Key string functions:

| Function | What it does | Example |
| --- | --- | --- |
| `split(col, pattern)` | Split string into array | `split("a,b,c", ",")` → `[a, b, c]` |
| `regexp_extract(col, pattern, group)` | Extract regex match | Pull email domain |
| `regexp_replace(col, pattern, replacement)` | Find & replace with regex | Clean phone numbers |
| `trim`, `ltrim`, `rtrim` | Remove whitespace | Clean user input |
| `initcap`, `upper`, `lower` | Case conversion | Standardize names |
| `lpad`, `rpad` | Pad to fixed width | Format IDs |
| `concat_ws(sep, cols...)` | Join columns with separator | Build full address |
| `substring(col, pos, len)` | Extract fixed-position text | Get year from date string |

In [0]:
# ============================================================
# STEP 7: String operations and regex
# ============================================================

# --- Create messy data ---
messy_data = [
    (1, "  John Smith  ", "john.smith@gmail.com", "(555) 123-4567", "Order#12345-A"),
    (2, "jane DOE", "jane_doe@company.co.uk", "555.987.6543", "Order#67890-B"),
    (3, "Bob Johnson III", "bob@yahoo.com", "5551112222", "Order#11111-C"),
    (4, " alice WONG ", "alice.wong@startup.io", "+1-555-000-9999", "Order#99999-D"),
]

df_messy = spark.createDataFrame(messy_data, ["id", "name", "email", "phone", "order_ref"])
print("MESSY DATA:")
df_messy.show(truncate=False)

# --- Clean it up ---
print("CLEANED:")
df_clean = df_messy.select(
    "id",
    # Trim whitespace + proper case
    initcap(trim(col("name"))).alias("name_clean"),
    
    # Extract email domain
    regexp_extract("email", r"@(.+)$", 1).alias("email_domain"),
    
    # Normalize phone: remove all non-digits
    regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    
    # Extract order number from reference
    regexp_extract("order_ref", r"Order#(\d+)", 1).cast("int").alias("order_num"),
    
    # Extract order suffix
    regexp_extract("order_ref", r"-(\w)$", 1).alias("order_suffix")
)
df_clean.show(truncate=False)

# --- Split example ---
print("SPLIT — break email into parts:")
df_messy.select(
    "email",
    split(col("email"), r"[@.]").alias("email_parts")
).show(truncate=False)

print("→ regex is essential for parsing log files, IDs, messy user input")

## 7.7 — Date & Timestamp Operations

### Key date functions:

| Function | What it does |
| --- | --- |
| `current_date()`, `current_timestamp()` | Now |
| `to_date(col, fmt)`, `to_timestamp(col, fmt)` | String → Date/Timestamp |
| `date_format(col, fmt)` | Date → formatted string |
| `datediff(end, start)` | Days between two dates |
| `months_between(end, start)` | Months between (decimal) |
| `date_add(col, days)`, `date_sub(col, days)` | Add/subtract days |
| `year(col)`, `month(col)`, `dayofweek(col)` | Extract parts |
| `last_day(col)` | Last day of month |
| `trunc(col, "month")` | Truncate to start of month/year |
| `from_utc_timestamp(col, tz)` | Convert UTC to timezone |

### Date format patterns:
* `yyyy-MM-dd` → 2024-03-15
* `dd/MM/yyyy` → 15/03/2024
* `yyyy-MM-dd HH:mm:ss` → 2024-03-15 14:30:00
* `EEE, MMM dd` → Fri, Mar 15

In [0]:
# ============================================================
# STEP 8: Date & timestamp operations
# ============================================================

# Use lineitem data for real date operations
df_dates = spark.read.table("samples.tpch.lineitem") \
    .select("l_orderkey", "l_shipdate", "l_commitdate", "l_receiptdate") \
    .limit(10)

print("1. DATE ARITHMETIC:")
df_dates.select(
    "l_orderkey",
    col("l_shipdate"),
    col("l_receiptdate"),
    datediff(col("l_receiptdate"), col("l_shipdate")).alias("delivery_days"),
    date_add(col("l_shipdate"), 30).alias("ship_plus_30"),
    months_between(col("l_receiptdate"), col("l_shipdate")).alias("months_diff")
).show()

print("2. DATE EXTRACTION:")
df_dates.select(
    "l_shipdate",
    year("l_shipdate").alias("year"),
    month("l_shipdate").alias("month"),
    dayofweek("l_shipdate").alias("day_of_week"),  # 1=Sun, 7=Sat
    date_format("l_shipdate", "EEEE").alias("day_name"),
    trunc("l_shipdate", "month").alias("month_start"),
    last_day("l_shipdate").alias("month_end")
).show()

# --- Parse messy date strings ---
print("3. PARSING different date formats:")
date_strings = [
    ("2024-03-15",),
    ("15/03/2024",),
    ("March 15, 2024",),
    ("20240315",),
]
df_str = spark.createDataFrame(date_strings, ["date_str"])

df_str.select(
    "date_str",
    expr("try_to_date(date_str, 'yyyy-MM-dd')").alias("fmt1"),
    expr("try_to_date(date_str, 'dd/MM/yyyy')").alias("fmt2"),
    expr("try_to_date(date_str, 'MMMM dd, yyyy')").alias("fmt3"),
    expr("try_to_date(date_str, 'yyyyMMdd')").alias("fmt4")
).show()
print("→ try_to_date returns NULL for non-matching formats (safe!)")
print("  to_date THROWS an error on mismatch. Always prefer try_to_date in production.")

print("\n4. TIMEZONE conversion:")
from pyspark.sql.functions import from_utc_timestamp, to_utc_timestamp

spark.sql("""
    SELECT 
        current_timestamp() AS utc_now,
        from_utc_timestamp(current_timestamp(), 'Asia/Kolkata') AS india_time,
        from_utc_timestamp(current_timestamp(), 'America/New_York') AS ny_time,
        from_utc_timestamp(current_timestamp(), 'Europe/London') AS london_time
""").show(truncate=False)

## Phase 7 Checkpoint

**Can you do these?**

- [ ] Create DataFrames with Struct, Array, and Map columns
- [ ] Access nested fields (dot notation, bracket notation, array index)
- [ ] Use `explode` / `explode_outer` / `posexplode` to flatten arrays
- [ ] Apply higher-order functions (`transform`, `filter`, `aggregate`) on arrays without exploding
- [ ] Pivot data (rows → columns) and unpivot (columns → rows)
- [ ] Parse raw JSON strings using `from_json` and `get_json_object`
- [ ] Clean messy text with `regexp_extract`, `regexp_replace`, `split`
- [ ] Do date arithmetic: `datediff`, `date_add`, `months_between`
- [ ] Parse different date formats with `to_date(col, format)`
- [ ] Convert between timezones with `from_utc_timestamp`

---

**Key takeaways:**
1. **Complex types** (Struct/Array/Map) are everywhere in real data — APIs, JSON, nested schemas
2. **Explode** turns one row into many — use `explode_outer` to keep nulls
3. **Higher-order functions** transform arrays without exploding (faster, simpler)
4. **Pivot/Unpivot** reshapes data between wide and long formats
5. **`from_json`** is the production pattern for parsing JSON columns
6. **Regex** is essential for cleaning log files, IDs, and user input
7. **Date functions** — always specify format explicitly; never rely on implicit parsing

---

**🎓 Congratulations! You’ve completed the PySpark Learning Series!**

This series (Phases 0–7) is the **pre-requisite** for the hands-on project course, which covers:
* Medallion Architecture (Bronze / Silver / Gold)
* Lakeflow Declarative Pipelines
* Jobs & Orchestration
* CI/CD with Declarative Automation Bundles
* Unity Catalog Governance